# 07_filter_evaluation — Second-session evaluation of the filters (psychophysics and fMRI)

**Manuscript:** Results section 7; Figure 7; Supplementary S14 (design), S15 (tab:exp2_8afc, tab:exp2_loro, tab:exp2_geometry), Figures S2-S3.

Both CVD participants repeated the JND and 8AFC tasks and were scanned under the deployed macOS accessibility filter and their individualized filter (four runs each, ABBA). `exp2_C010_conditions.py` estimates the condition-wise amplitudes; `exp2_hc_likeness.py` and `exp2_decoder_2x2.py` compute LORO / LOCO readouts against the run-matched control reference; `exp2_runmatched_geometry.py` rebuilds SRM disparity and RDM similarity over all C(6,4) run subsets; `analyze_exp2_behavior.py` and `analyze_exp2_jnd_vs_hc.py` handle the psychophysics; `run_count_adjacc.py` is the run-count adequacy analysis (Figure S2). All neural indices are single-case descriptive (d_cc); the notebook loads the committed condition-level outputs.

**How to read this notebook.** Every code cell loads committed result files from `results/` and compares the values it derives with the numbers printed in the manuscript (`V.check`). A check passes when the produced value equals the printed one at the printed precision, or satisfies the stated relation. Quantities that have no committed artifact are recorded as pointers (`V.flag`) rather than silently omitted. The last cell tallies the checks and writes `_checks_07_filter_evaluation.json`, which `run_notebooks.py` collects into `REPORT.md`.

Provenance: built by `tools/public_repo/build.py` of the development repository (commit 53c81c2); manuscript source in `../paper/`; check list in `../MANIFEST.md`; code map in `../MAP.md`.

**Source and code map**

| Result file | Producing script | What it holds |
|---|---|---|
| `results/behavior/sub-0{8,9}_summary.json` | `scripts/analyze_exp2_behavior.py` | JND means per condition, Wilcoxon between filters, 8AFC counts |
| `results/behavior/sub-0{8,9}_jnd_vs_hc.json` | `scripts/analyze_exp2_jnd_vs_hc.py` | z of every condition's threshold against the controls, per pair |
| `results/neural/exp2_hc_likeness_sub-0{8,9}_matched.json` | `scripts/exp2_hc_likeness.py` | LORO / LOCO readouts per condition vs the run-matched control reference (tab:exp2_loro, tab:exp2_geometry, Figure S3) |
| `results/neural/exp2_runmatched_geometry_sub-0{8,9}_matched.json` | `scripts/exp2_runmatched_geometry.py` | SRM disparity and RDM similarity, all C(6,4) subsets (tab:exp2_geometry) |
| `results/neural/exp2_convergent_sub-09_matched.json` | `scripts/exp2_convergent.py` | unmatched geometry (S14 matching comparison) |
| `results/adjacc_retention_summary.json` | `scripts/run_count_adjacc.py` | run-count adequacy (S14, Figure S2) |

In [1]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / ".." / "common").resolve()))
import numpy as np
from scipy import stats
import verify as V
from stats_helpers import crawford_howell, hedges_g, bh_fdr, wilson_interval
R = Path("results")
def J(name):
    with open(R / name) as f:
        return json.load(f)
HC = [f"sub-{i:02d}" for i in range(1, 8)]
CVD = {"deutan": "sub-08", "protan": "sub-09"}
ROIS = ["V1", "V2", "V3", "hV4"]
HUES = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "magenta"]

B = {k: J(f"behavior/{s}_summary.json") for k, s in CVD.items()}
Z = {k: J(f"behavior/{s}_jnd_vs_hc.json") for k, s in CVD.items()}
HL = {k: J(f"neural/exp2_hc_likeness_{s}_matched.json") for k, s in CVD.items()}
RM = {k: J(f"neural/exp2_runmatched_geometry_{s}_matched.json")["rois"] for k, s in CVD.items()}
COND = {"NF": "nofilter", "Dep": "window", "Ind": "optimal"}
RK = {"V1": "V1", "V2": "V2", "V3": "V3", "hV4": "V4"}
PAIRS = ["orange-yellow", "yellow-green", "green-blue", "cyan-magenta", "yellow-purple", "blue-purple", "red-orange", "red-cyan"]

V.start("07_filter_evaluation")

### Psychophysics under each filter (Results section 7 'Psychophysics')
z of each threshold against the control distribution; NF = session-1 baseline, Dep = deployed accessibility filter, Ind = individualized filter.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 07.01 | Results §7 | deutan mean |z| at baseline | `2.24` |
| 07.02 | Results §7 | deutan mean |z| under the deployed filter (below 0.9) | `0.9` |
| 07.03 | Results §7 | deutan mean |z| under the individualized filter (below 0.9) | `0.9` |
| 07.04 | Results §7 | deutan: all three elevated pairs within ±1.8 control SD under each filter | `1.8` |
| 07.05 | Results §7 | deutan Wilcoxon between the two filters p = 0.84 | `0.84` |
| 07.06 | Results §7 | protan mean |z| at baseline | `0.9` |
| 07.07 | Results §7 | protan mean |z| under the deployed filter | `1.78` |
| 07.08 | Results §7 | protan mean |z| under the individualized filter | `0.93` |
| 07.09 | Results §7 | deployed filter left two protan pairs deviant (|z| > 2) | `2` |
| 07.10 | Results §7 | individualized filter held every protan pair within ±1.5 | `1.5` |
| 07.11 | Results §7 | only the individualized filter kept every previously non-deviant pair inside the control range in both participants | `(True, False)` |
| 07.12 | Abstract | the four session-1 elevations (three deutan, one protan) moved inside the control range under the individualized filter | `2.0` |

In [2]:
zz = {k: {c: {p: Z[k]["crawford_howell"][p][COND[c]]["z_cc"] for p in PAIRS} for c in COND} for k in CVD}
mz = {k: {c: Z[k]["mean_abs_z_to_HC"][COND[c]] for c in COND} for k in CVD}
elev8 = ["orange-yellow", "yellow-green", "yellow-purple"]
elev8_max = max(abs(zz["deutan"][c][p]) for c in ("Dep", "Ind") for p in elev8)
dep9_dev = [p for p in PAIRS if abs(zz["protan"]["Dep"][p]) > 2]
ind9_max = max(abs(zz["protan"]["Ind"][p]) for p in PAIRS)
def kept(k, c):
    return all(abs(zz[k][c][p]) < 2 for p in PAIRS if abs(zz[k]["NF"][p]) < 2)
print(mz, elev8_max, dep9_dev, ind9_max, {k: {c: kept(k, c) for c in ("Dep", "Ind")} for k in CVD})
V.check('07.01', 'Results §7 | deutan mean |z| at baseline', mz["deutan"]["NF"], 2.24, nd=2)
V.check('07.02', 'Results §7 | deutan mean |z| under the deployed filter (below 0.9)', mz["deutan"]["Dep"], 0.9, mode='lt')
V.check('07.03', 'Results §7 | deutan mean |z| under the individualized filter (below 0.9)', mz["deutan"]["Ind"], 0.9, mode='lt')
V.check('07.04', 'Results §7 | deutan: all three elevated pairs within ±1.8 control SD under each filter', elev8_max, 1.8, mode='le')
V.check('07.05', 'Results §7 | deutan Wilcoxon between the two filters p = 0.84', B["deutan"]["wilcoxon"]["wilcoxon_opt_vs_win"]["p"], 0.84, nd=2)
V.check('07.06', 'Results §7 | protan mean |z| at baseline', mz["protan"]["NF"], 0.9, nd=2)
V.check('07.07', 'Results §7 | protan mean |z| under the deployed filter', mz["protan"]["Dep"], 1.78, nd=2)
V.check('07.08', 'Results §7 | protan mean |z| under the individualized filter', mz["protan"]["Ind"], 0.93, nd=2)
V.check('07.09', 'Results §7 | deployed filter left two protan pairs deviant (|z| > 2)', len(dep9_dev), 2, mode='eq')
V.check('07.10', 'Results §7 | individualized filter held every protan pair within ±1.5', ind9_max, 1.5, mode='le')
V.check('07.11', 'Results §7 | only the individualized filter kept every previously non-deviant pair inside the control range in both participants', (kept("deutan", "Ind") and kept("protan", "Ind"), kept("deutan", "Dep") and kept("protan", "Dep")), (True, False), mode='eq')
V.check('07.12', 'Abstract | the four session-1 elevations (three deutan, one protan) moved inside the control range under the individualized filter', max([abs(zz["deutan"]["Ind"][p]) for p in elev8] + [abs(zz["protan"]["Ind"]["green-blue"])]), 2.0, mode='lt')

{'deutan': {'NF': 2.2407, 'Dep': 0.8527, 'Ind': 0.7798}, 'protan': {'NF': 0.8966, 'Dep': 1.7752, 'Ind': 0.9344}} 1.717 ['green-blue', 'cyan-magenta'] 1.495 {'deutan': {'Dep': True, 'Ind': True}, 'protan': {'Dep': False, 'Ind': True}}
[OK ] 07.01 Results §7 | deutan mean |z| at baseline: produced=2.241  reported=2.24
[OK ] 07.02 Results §7 | deutan mean |z| under the deployed filter (below 0.9): produced=0.8527  reported=0.9
[OK ] 07.03 Results §7 | deutan mean |z| under the individualized filter (below 0.9): produced=0.7798  reported=0.9
[OK ] 07.04 Results §7 | deutan: all three elevated pairs within ±1.8 control SD under each filter: produced=1.717  reported=1.8
[OK ] 07.05 Results §7 | deutan Wilcoxon between the two filters p = 0.84: produced=0.8438  reported=0.84
[OK ] 07.06 Results §7 | protan mean |z| at baseline: produced=0.8966  reported=0.9
[OK ] 07.07 Results §7 | protan mean |z| under the deployed filter: produced=1.775  reported=1.78
[OK ] 07.08 Results §7 | protan mean |z

### Identification accuracy (Results section 7; Supplementary tab:exp2_8afc)
8AFC accuracy over 64 trials per cell with the 95% Wilson score interval.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 07.13 | tab:exp2_8afc | 2 participants x 3 conditions x (accuracy, CI, n) | `24 cells, see 07.T1.*` |

In [3]:
T = {"deutan": {"NF": (0.81, 0.70, 0.89), "Dep": (0.97, 0.89, 0.99), "Ind": (0.97, 0.89, 0.99)},
     "protan": {"NF": (1.00, 0.94, 1.00), "Dep": (0.86, 0.75, 0.92), "Ind": (0.98, 0.92, 1.00)}}
KEY = {"NF": "baseline", "Dep": "window", "Ind": "optimal"}
for k in CVD:
    for c, (acc_r, lo_r, hi_r) in T[k].items():
        r = B[k]["rsvp"][KEY[c]]; lo, hi = wilson_interval(r["n_correct"], r["n"])
        V.check(f"07.T1.{k}.{c}.acc", f"tab:exp2_8afc {k} {c} accuracy", r["acc"], acc_r, nd=2)
        V.check(f"07.T1.{k}.{c}.lo", f"tab:exp2_8afc {k} {c} CI low", lo, lo_r, nd=2)
        V.check(f"07.T1.{k}.{c}.hi", f"tab:exp2_8afc {k} {c} CI high", hi, hi_r, nd=2)
        V.check(f"07.T1.{k}.{c}.n", f"tab:exp2_8afc {k} {c} n = 64", r["n"], 64, mode="eq")
V.table('07.13', 'tab:exp2_8afc | 2 participants x 3 conditions x (accuracy, CI, n)', '24 cells, see 07.T1.*')

[OK ] 07.T1.deutan.NF.acc tab:exp2_8afc deutan NF accuracy: produced=0.8125  reported=0.81
[OK ] 07.T1.deutan.NF.lo tab:exp2_8afc deutan NF CI low: produced=0.7003  reported=0.7
[OK ] 07.T1.deutan.NF.hi tab:exp2_8afc deutan NF CI high: produced=0.8894  reported=0.89
[OK ] 07.T1.deutan.NF.n tab:exp2_8afc deutan NF n = 64: produced=64  reported=64
[OK ] 07.T1.deutan.Dep.acc tab:exp2_8afc deutan Dep accuracy: produced=0.9688  reported=0.97
[OK ] 07.T1.deutan.Dep.lo tab:exp2_8afc deutan Dep CI low: produced=0.893  reported=0.89
[OK ] 07.T1.deutan.Dep.hi tab:exp2_8afc deutan Dep CI high: produced=0.9914  reported=0.99
[OK ] 07.T1.deutan.Dep.n tab:exp2_8afc deutan Dep n = 64: produced=64  reported=64
[OK ] 07.T1.deutan.Ind.acc tab:exp2_8afc deutan Ind accuracy: produced=0.9688  reported=0.97
[OK ] 07.T1.deutan.Ind.lo tab:exp2_8afc deutan Ind CI low: produced=0.893  reported=0.89
[OK ] 07.T1.deutan.Ind.hi tab:exp2_8afc deutan Ind CI high: produced=0.9914  reported=0.99
[OK ] 07.T1.deutan.Ind.

### Hues remained decodable (Results section 7; Supplementary tab:exp2_loro)
LORO eight-way accuracy per ROI and condition against the control mean.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 07.14 | tab:exp2_loro | 4 ROIs x (controls + 2 participants x 3 conditions) | `28 cells, see 07.T2.*` |
| 07.15 | Results §7 | lowest cell | `0.5` |
| 07.16 | Results §7 | control range low | `0.71` |
| 07.17 | Results §7 | control range high | `0.77` |
| 07.18 | Results §7 | every cell above chance 0.125 | `0.125` |

In [4]:
T = {"V1": (0.71, 0.79, 0.84, 0.72, 0.79, 0.91, 0.66), "V2": (0.71, 0.79, 0.69, 0.62, 0.83, 0.75, 0.72),
     "V3": (0.77, 0.67, 0.78, 0.69, 0.81, 1.00, 0.69), "hV4": (0.75, 0.73, 0.88, 0.50, 0.71, 0.84, 0.69)}
loro_cells = []
for roi, vals in T.items():
    h8 = HL["deutan"][RK[roi]]; h9 = HL["protan"][RK[roi]]
    got = (h8["hc_loro_acc_mean"], h8["nofilter_baseline_exp1"]["loro_acc"], h8["conditions"]["window"]["loro_acc"], h8["conditions"]["optimal"]["loro_acc"],
           h9["nofilter_baseline_exp1"]["loro_acc"], h9["conditions"]["window"]["loro_acc"], h9["conditions"]["optimal"]["loro_acc"])
    for lab, g, rep in zip(("controls", "deutan NF", "deutan Dep", "deutan Ind", "protan NF", "protan Dep", "protan Ind"), got, vals):
        V.check(f"07.T2.{roi}.{lab}", f"tab:exp2_loro {roi} {lab}", g, rep, nd=2)
    loro_cells.extend(got[1:])
hc_range = (min(HL["deutan"][RK[r]]["hc_loro_acc_mean"] for r in ROIS), max(HL["deutan"][RK[r]]["hc_loro_acc_mean"] for r in ROIS))
print(min(loro_cells), hc_range)
V.table('07.14', 'tab:exp2_loro | 4 ROIs x (controls + 2 participants x 3 conditions)', '28 cells, see 07.T2.*')
V.check('07.15', 'Results §7 | lowest cell', min(loro_cells), 0.5, nd=2)
V.check('07.16', 'Results §7 | control range low', hc_range[0], 0.71, nd=2)
V.check('07.17', 'Results §7 | control range high', hc_range[1], 0.77, nd=2)
V.check('07.18', 'Results §7 | every cell above chance 0.125', min(loro_cells), 0.125, mode='gt')

[OK ] 07.T2.V1.controls tab:exp2_loro V1 controls: produced=0.7143  reported=0.71
[OK ] 07.T2.V1.deutan NF tab:exp2_loro V1 deutan NF: produced=0.7917  reported=0.79
[OK ] 07.T2.V1.deutan Dep tab:exp2_loro V1 deutan Dep: produced=0.8438  reported=0.84
[OK ] 07.T2.V1.deutan Ind tab:exp2_loro V1 deutan Ind: produced=0.7188  reported=0.72
[OK ] 07.T2.V1.protan NF tab:exp2_loro V1 protan NF: produced=0.7917  reported=0.79
[OK ] 07.T2.V1.protan Dep tab:exp2_loro V1 protan Dep: produced=0.9062  reported=0.91
[OK ] 07.T2.V1.protan Ind tab:exp2_loro V1 protan Ind: produced=0.6562  reported=0.66
[OK ] 07.T2.V2.controls tab:exp2_loro V2 controls: produced=0.7054  reported=0.71
[OK ] 07.T2.V2.deutan NF tab:exp2_loro V2 deutan NF: produced=0.7917  reported=0.79
[OK ] 07.T2.V2.deutan Dep tab:exp2_loro V2 deutan Dep: produced=0.6875  reported=0.69
[OK ] 07.T2.V2.deutan Ind tab:exp2_loro V2 deutan Ind: produced=0.625  reported=0.62
[OK ] 07.T2.V2.protan NF tab:exp2_loro V2 protan NF: produced=0.8333 

### Interpolation and geometry by condition (Results section 7; Figure 7; Supplementary tab:exp2_geometry)
Run-matched (four runs per cell). Parenthetical values in the table are Crawford-Howell d_cc against the control distribution.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 07.19 | tab:exp2_geometry | adjacent accuracy, disparity and RDM similarity by condition with d_cc | `42 cells, see 07.T3.*` |
| 07.20 | Results §7 'Interpolation' | deutan: individualized filter is the only condition above chance 0.25 at hV4 | `['Ind']` |
| 07.21 | Results §7 'Interpolation' | protan: individualized filter is the lowest of the three, all below chance | `('Ind', True)` |
| 07.22 | Results §7 'Geometry' | deutan: both filters raise V2 disparity above baseline and lower RDM similarity | `True` |
| 07.23 | Results §7 'Geometry' | protan: both filters reduce V1 disparity; RDM similarity rises under Dep and falls under Ind | `(True, True, True)` |

In [5]:
TARGET = {"deutan": "V2", "protan": "V1"}
T_ADJ = {"deutan": ((0.23, -2.11), (0.25, -1.94), (0.31, -1.35)), "protan": ((0.14, -2.99), (0.19, -2.52), (0.06, -3.70))}
T_DISP = {"deutan": (0.44, (0.68, 2.69), (0.87, 4.93), (0.77, 3.73)), "protan": (0.43, (0.70, 3.92), (0.66, 3.29), (0.63, 2.84))}
T_RDM = {"deutan": (0.59, 0.42, 0.16, 0.05), "protan": (0.66, 0.33, 0.38, 0.26)}
adj = {}; disp = {}; rdm = {}
for k in CVD:
    h = HL[k]["V4"]
    adj[k] = {"NF": (h["nofilter_baseline_exp1"]["loco_adjacc_n4_matched"], h["nofilter_baseline_exp1"]["loco_adjacc_d_vs_hc_n4"]),
              "Dep": (h["conditions"]["window"]["loco_adjacc_mean"], h["conditions"]["window"]["loco_adjacc_d_vs_hc_n4matched"]),
              "Ind": (h["conditions"]["optimal"]["loco_adjacc_mean"], h["conditions"]["optimal"]["loco_adjacc_d_vs_hc_n4matched"])}
    V.check(f"07.T3.{k}.hV4.controls_mean", f"tab:exp2_geometry {k} hV4 controls mean", h["hc_loco_adjacc_n4_mean"], 0.46, nd=2)
    V.check(f"07.T3.{k}.hV4.controls_sd", f"tab:exp2_geometry {k} hV4 controls SD", h["hc_loco_adjacc_n4_sd"], 0.11, nd=2)
    V.check(f"07.T3.{k}.hV4.controls_n", f"tab:exp2_geometry {k} hV4 n = 6 controls", h["hc_n"], 6, mode="eq")
    for c, (a_r, d_r) in zip(("NF", "Dep", "Ind"), T_ADJ[k]):
        V.check(f"07.T3.{k}.hV4.{c}.adj", f"tab:exp2_geometry {k} hV4 adjacent {c}", adj[k][c][0], a_r, nd=2)
        V.check(f"07.T3.{k}.hV4.{c}.d", f"tab:exp2_geometry {k} hV4 adjacent {c} d_cc", adj[k][c][1], d_r, nd=2)
    g = RM[k][TARGET[k]]
    disp[k] = {c: (g["srm"]["conditions"][COND[c]]["disparity"], g["srm"]["conditions"][COND[c]]["d"]) for c in COND}
    rdm[k] = {c: g["srm_rdm_paper"][COND[c]]["spearman_to_hc"] for c in COND}
    V.check(f"07.T3.{k}.{TARGET[k]}.disp.controls", f"tab:exp2_geometry {k} {TARGET[k]} disparity controls", g["srm"]["hc_disp_mean"], T_DISP[k][0], nd=2)
    for c, (x_r, d_r) in zip(("NF", "Dep", "Ind"), T_DISP[k][1:]):
        V.check(f"07.T3.{k}.{TARGET[k]}.disp.{c}", f"tab:exp2_geometry {k} {TARGET[k]} disparity {c}", disp[k][c][0], x_r, nd=2)
        V.check(f"07.T3.{k}.{TARGET[k]}.disp.{c}.d", f"tab:exp2_geometry {k} {TARGET[k]} disparity {c} d_cc", disp[k][c][1], d_r, nd=2)
    V.check(f"07.T3.{k}.{TARGET[k]}.rdm.controls", f"tab:exp2_geometry {k} {TARGET[k]} RDM similarity controls (LOO self-consistency)", g["srm_rdm_paper"]["_hc"]["spearman_self_loo_mean"], T_RDM[k][0], nd=2)
    for c, x_r in zip(("NF", "Dep", "Ind"), T_RDM[k][1:]):
        V.check(f"07.T3.{k}.{TARGET[k]}.rdm.{c}", f"tab:exp2_geometry {k} {TARGET[k]} RDM similarity {c}", rdm[k][c], x_r, nd=2)
    V.check(f"07.T3.{k}.n_subsets", f"S14 {k} C(6,4) = 15 subsets, n = 7 controls", (g["n_subsets"], g["n_hc"]), (15, 7), mode="eq")
print(adj, disp, rdm)
V.table('07.19', 'tab:exp2_geometry | adjacent accuracy, disparity and RDM similarity by condition with d_cc', '42 cells, see 07.T3.*')
V.check('07.20', "Results §7 'Interpolation' | deutan: individualized filter is the only condition above chance 0.25 at hV4", [c for c in adj["deutan"] if adj["deutan"][c][0] > 0.25], ['Ind'], mode='eq')
V.check('07.21', "Results §7 'Interpolation' | protan: individualized filter is the lowest of the three, all below chance", (min(adj["protan"], key=lambda c: adj["protan"][c][0]), max(v[0] for v in adj["protan"].values()) < 0.25), ('Ind', True), mode='eq')
V.check('07.22', "Results §7 'Geometry' | deutan: both filters raise V2 disparity above baseline and lower RDM similarity", all(disp["deutan"][c][0] > disp["deutan"]["NF"][0] and rdm["deutan"][c] < rdm["deutan"]["NF"] for c in ("Dep", "Ind")), True, mode='eq')
V.check('07.23', "Results §7 'Geometry' | protan: both filters reduce V1 disparity; RDM similarity rises under Dep and falls under Ind", (all(disp["protan"][c][0] < disp["protan"]["NF"][0] for c in ("Dep", "Ind")), rdm["protan"]["Dep"] > rdm["protan"]["NF"], rdm["protan"]["Ind"] < rdm["protan"]["NF"]), (True, True, True), mode='eq')

[OK ] 07.T3.deutan.hV4.controls_mean tab:exp2_geometry deutan hV4 controls mean: produced=0.4556  reported=0.46
[OK ] 07.T3.deutan.hV4.controls_sd tab:exp2_geometry deutan hV4 controls SD: produced=0.1062  reported=0.11
[OK ] 07.T3.deutan.hV4.controls_n tab:exp2_geometry deutan hV4 n = 6 controls: produced=6  reported=6
[OK ] 07.T3.deutan.hV4.NF.adj tab:exp2_geometry deutan hV4 adjacent NF: produced=0.2313  reported=0.23
[OK ] 07.T3.deutan.hV4.NF.d tab:exp2_geometry deutan hV4 adjacent NF d_cc: produced=-2.111  reported=-2.11
[OK ] 07.T3.deutan.hV4.Dep.adj tab:exp2_geometry deutan hV4 adjacent Dep: produced=0.25  reported=0.25
[~~ ] 07.T3.deutan.hV4.Dep.d tab:exp2_geometry deutan hV4 adjacent Dep d_cc: produced=-1.935  reported=-1.94
[OK ] 07.T3.deutan.hV4.Ind.adj tab:exp2_geometry deutan hV4 adjacent Ind: produced=0.3125  reported=0.31
[OK ] 07.T3.deutan.hV4.Ind.d tab:exp2_geometry deutan hV4 adjacent Ind d_cc: produced=-1.347  reported=-1.35
[OK ] 07.T3.deutan.V2.disp.controls tab:ex

### Forward-tuning encoding index (Supplementary S15 last paragraph; Figure S3)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 07.24 | S15 'Forward-tuning' | deutan hV4 rho under the individualized filter | `0.18` |
| 07.25 | S15 | control rho at hV4 | `0.21` |
| 07.26 | S15 | deutan hV4 rho under the deployed filter | `-0.39` |
| 07.27 | S15 | protan: three conditions all near -0.02 (|rho + 0.02| < 0.01) | `0.01` |

In [6]:
rho = {k: dict(hc=HL[k]["V4"]["hc_loco_rho_n4_mean"], Dep=HL[k]["V4"]["conditions"]["window"]["loco_rho_mean"], Ind=HL[k]["V4"]["conditions"]["optimal"]["loco_rho_mean"], NF=HL[k]["V4"]["nofilter_baseline_exp1"]["loco_rho_n4_matched"]) for k in CVD}
print(rho)
V.check('07.24', "S15 'Forward-tuning' | deutan hV4 rho under the individualized filter", rho["deutan"]["Ind"], 0.18, nd=2)
V.check('07.25', 'S15 | control rho at hV4', rho["deutan"]["hc"], 0.21, nd=2)
V.check('07.26', 'S15 | deutan hV4 rho under the deployed filter', rho["deutan"]["Dep"], -0.39, nd=2)
V.check('07.27', 'S15 | protan: three conditions all near -0.02 (|rho + 0.02| < 0.01)', max(abs(rho["protan"][c] + 0.02) for c in ("NF", "Dep", "Ind")), 0.01, mode='lt')

{'deutan': {'hc': 0.20789970301206742, 'Dep': -0.38821069100812833, 'Ind': 0.17886295791630277, 'NF': -0.27232845389474786}, 'protan': {'hc': 0.20789970301206742, 'Dep': -0.017909593468569927, 'Ind': -0.02401360647315902, 'NF': -0.020719781391996754}}
[OK ] 07.24 S15 'Forward-tuning' | deutan hV4 rho under the individualized filter: produced=0.1789  reported=0.18
[OK ] 07.25 S15 | control rho at hV4: produced=0.2079  reported=0.21
[OK ] 07.26 S15 | deutan hV4 rho under the deployed filter: produced=-0.3882  reported=-0.39
[OK ] 07.27 S15 | protan: three conditions all near -0.02 (|rho + 0.02| < 0.01): produced=0.004014  reported=0.01


### Run-count adequacy and run matching (Supplementary S14)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 07.28 | S14 'Run-count adequacy' | hV4 control mean at four runs | `0.45` |
| 07.29 | S14 | at six runs | `0.46` |
| 07.30 | S14 | deutan at four runs | `0.23` |
| 07.31 | S14 | protan at four runs | `0.14` |
| 07.32 | S14 | single-case d_cc < -2 in both at four runs | `-2.0` |
| 07.33 | S14 | the control-CVD separation holds at every run count down to four (control mean above both CVD cases and above chance) | `True` |
| 07.33b | S14 | both CVD participants below chance at four runs | `0.25` |
| 07.34 | S14 | protan V1 RDM similarity of the unfiltered baseline, unmatched | `0.25` |
| 07.35 | S14 | same after run matching | `0.33` |
| 07.36 | S14 | four runs per filter condition | `(4, 4)` |

In [7]:
rc = J("adjacc_retention_summary.json")["per_roi"]["V4"]
n4 = rc["4"]; n6 = rc["6"]
cv9 = J("neural/exp2_convergent_sub-09_matched.json")["V1"]["srm_rdm_paper"]["nofilter"]["spearman_to_hc"]
print(n4["hc_mean"], n6["hc_mean"], n4["cvd"]["08"], n4["cvd"]["09"], cv9, RM["protan"]["V1"]["srm_rdm_paper"]["nofilter"]["spearman_to_hc"])
V.check('07.28', "S14 'Run-count adequacy' | hV4 control mean at four runs", n4["hc_mean"], 0.45, nd=2)
V.check('07.29', 'S14 | at six runs', n6["hc_mean"], 0.46, nd=2)
V.check('07.30', 'S14 | deutan at four runs', n4["cvd"]["08"]["adjacc"], 0.23, nd=2)
V.check('07.31', 'S14 | protan at four runs', n4["cvd"]["09"]["adjacc"], 0.14, nd=2)
V.check('07.32', 'S14 | single-case d_cc < -2 in both at four runs', max(n4["cvd"]["08"]["d_cc"], n4["cvd"]["09"]["d_cc"]), -2.0, mode='lt')
V.check('07.33', 'S14 | the control-CVD separation holds at every run count down to four (control mean above both CVD cases and above chance)', all(rc[n]["hc_mean"] > 0.25 and rc[n]["hc_mean"] > max(rc[n]["cvd"]["08"]["adjacc"], rc[n]["cvd"]["09"]["adjacc"]) for n in ("4", "5", "6")), True, mode='eq')
V.check('07.33b', 'S14 | both CVD participants below chance at four runs', max(n4["cvd"]["08"]["adjacc"], n4["cvd"]["09"]["adjacc"]), 0.25, mode='lt')
V.check('07.34', 'S14 | protan V1 RDM similarity of the unfiltered baseline, unmatched', cv9, 0.25, nd=2)
V.check('07.35', 'S14 | same after run matching', RM["protan"]["V1"]["srm_rdm_paper"]["nofilter"]["spearman_to_hc"], 0.33, nd=2)
V.check('07.36', 'S14 | four runs per filter condition', (HL["deutan"]["V4"]["conditions"]["window"]["n_runs"], HL["deutan"]["V4"]["conditions"]["optimal"]["n_runs"]), (4, 4), mode='eq')

0.4491071428571428 0.45595238095238094 {'adjacc': 0.23125, 'd_cc': -2.212, 'p': 0.042, 'below_chance': True} {'adjacc': 0.1375, 'd_cc': -3.165, 'p': 0.0126, 'below_chance': True} 0.2517788724685276 0.33231162196679437
[OK ] 07.28 S14 'Run-count adequacy' | hV4 control mean at four runs: produced=0.4491  reported=0.45
[OK ] 07.29 S14 | at six runs: produced=0.456  reported=0.46
[OK ] 07.30 S14 | deutan at four runs: produced=0.2313  reported=0.23
[OK ] 07.31 S14 | protan at four runs: produced=0.1375  reported=0.14
[OK ] 07.32 S14 | single-case d_cc < -2 in both at four runs: produced=-2.212  reported=-2
[OK ] 07.33 S14 | the control-CVD separation holds at every run count down to four (control mean above both CVD cases and above chance): produced=True  reported=True
[OK ] 07.33b S14 | both CVD participants below chance at four runs: produced=0.2313  reported=0.25
[OK ] 07.34 S14 | protan V1 RDM similarity of the unfiltered baseline, unmatched: produced=0.2518  reported=0.25
[OK ] 07.35

In [8]:
V.summary()


=== 07_filter_evaluation: 127/128 numeric checks reproduced exactly; 1 within one unit of the last printed digit; 0 mismatch, 0 error, 0 pointer-only ===
  NEAR     07.T3.deutan.hV4.Dep.d tab:exp2_geometry deutan hV4 adjacent Dep d_cc: produced=-1.935 reported=-1.94
